# 05 — Baseline Sanity Check

Verify the full training pipeline end-to-end **without** running a real training job.

Checks:
1. Config loads, model instantiates, architecture summary + param count
2. One forward pass produces correct output shape `(B, 234)`
3. BCE loss computes without errors
4. 2 mini-epochs on 50 samples — loss decreases, AUC is computed

In [1]:
import os
os.chdir('..')
print('Working dir:', os.getcwd())

import sys, time
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from pathlib import Path
from torch.utils.data import DataLoader, Subset

from src.training.train import BirdCLEFModel, get_target_species
from src.data.dataset import BirdCLEFDataset
from src.utils.metrics import macro_auc
from src.utils.seed import seed_everything

Working dir: /home/gokhuu/Portfolio/Kaggle/BirdCLEF


## 1 — Load config, build model, print summary

In [5]:
CONFIG_PATH = 'configs/baseline.yaml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('Config loaded:')
for k, v in cfg.items():
    print(f'  {k}: {v}')

seed_everything(cfg['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

Config loaded:
  run_name: baseline_effb0_fold0
  seed: 42
  fold: 0
  folds_csv: data/folds/folds.csv
  spec_dir: data/processed
  train_meta_csv: data/raw/train.csv
  audio_dir: data/raw/train_audio
  num_workers: 4
  backbone: tf_efficientnet_b0_ns
  num_classes: 234
  dropout: 0.3
  pretrained: True
  epochs: 30
  batch_size: 32
  lr: 0.001
  weight_decay: 0.0001
  scheduler: cosine
  warmup_epochs: 2
  spec_augment: True
  time_mask_max: 50
  freq_mask_max: 20
  mixup_alpha: 0.4
  gaussian_noise_std: 0.01

Device: cuda
  GPU: NVIDIA GeForce RTX 2060 with Max-Q Design


In [6]:
model = BirdCLEFModel(
    backbone=cfg['backbone'],
    num_classes=cfg['num_classes'],
    dropout=cfg.get('dropout', 0.3),
    pretrained=cfg.get('pretrained', True),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {n_params:>12,}')
print(f'Trainable parameters: {n_trainable:>12,}')
print()
print(model)

/home/gokhuu/miniconda3/envs/birdclef/lib/python3.11/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


Total parameters:        4,306,726
Trainable parameters:    4,306,726

BirdCLEFModel(
  (encoder): EfficientNet(
    (conv_stem): Conv2dSame(1, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
    

## 2 — Forward pass shape check

In [7]:
B = cfg['batch_size']
dummy_input = torch.randn(B, 1, 128, 313).to(device)  # (B, 1, n_mels, T)

model.eval()
with torch.no_grad():
    logits = model(dummy_input)

expected_shape = (B, cfg['num_classes'])
assert logits.shape == expected_shape, f'Got {logits.shape}, expected {expected_shape}'
print(f'Output shape: {logits.shape}  ✓')
print(f'Logits range: [{logits.min().item():.3f}, {logits.max().item():.3f}]')

Output shape: torch.Size([32, 234])  ✓
Logits range: [-0.410, 0.412]


## 3 — Loss computation

In [8]:
criterion = nn.BCEWithLogitsLoss()
dummy_target = torch.zeros(B, cfg['num_classes']).to(device)
for i in range(B):
    n_active = np.random.randint(1, 4)
    cols = np.random.choice(cfg['num_classes'], size=n_active, replace=False)
    dummy_target[i, cols] = 1.0

loss = criterion(logits, dummy_target)
print(f'BCE loss: {loss.item():.4f}  ✓')

# Verify backward works
model.train()
logits2 = model(dummy_input)
loss2 = criterion(logits2, dummy_target)
loss2.backward()
print('Backward pass: ✓')

BCE loss: 0.6979  ✓
Backward pass: ✓


## 4 — Mini-training: 2 epochs on 50 samples

In [9]:
target_species = get_target_species(cfg)
print(f'Target species: {len(target_species)}')

ds_full = BirdCLEFDataset(
    folds_csv=cfg['folds_csv'], fold=cfg['fold'], mode='train',
    spec_dir=cfg['spec_dir'], target_species=target_species,
    train_meta_csv=cfg['train_meta_csv'], audio_dir=cfg['audio_dir'],
    aug_spec_p=0.0, aug_noise_p=0.0, aug_mixup_p=0.0,
)
ds_val_full = BirdCLEFDataset(
    folds_csv=cfg['folds_csv'], fold=cfg['fold'], mode='val',
    spec_dir=cfg['spec_dir'], target_species=target_species,
    train_meta_csv=cfg['train_meta_csv'], audio_dir=cfg['audio_dir'],
)

N_SAMPLES = min(50, len(ds_full))
N_VAL = min(50, len(ds_val_full))
ds_tiny_train = Subset(ds_full, list(range(N_SAMPLES)))
ds_tiny_val = Subset(ds_val_full, list(range(N_VAL)))

loader_train = DataLoader(ds_tiny_train, batch_size=16, shuffle=True, num_workers=0)
loader_val = DataLoader(ds_tiny_val, batch_size=16, shuffle=False, num_workers=0)

print(f'Mini-train: {len(ds_tiny_train)} samples, Mini-val: {len(ds_tiny_val)} samples')

Target species: 234
Mini-train: 50 samples, Mini-val: 50 samples


In [10]:
seed_everything(cfg['seed'])
model_mini = BirdCLEFModel(
    backbone=cfg['backbone'],
    num_classes=cfg['num_classes'],
    dropout=cfg.get('dropout', 0.3),
    pretrained=cfg.get('pretrained', True),
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model_mini.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

losses = []
for epoch in range(1, 3):
    model_mini.train()
    epoch_loss = 0.0
    n_batch = 0
    for x, y in loader_train:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model_mini(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batch += 1
    avg_loss = epoch_loss / n_batch
    losses.append(avg_loss)

    # Quick val AUC
    model_mini.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for x, y in loader_val:
            x = x.to(device)
            preds = torch.sigmoid(model_mini(x)).cpu().numpy()
            all_preds.append(preds)
            all_trues.append(y.numpy())
    all_preds = np.concatenate(all_preds)
    all_trues = np.concatenate(all_trues)
    auc_val, _ = macro_auc(all_trues, all_preds, target_species)
    print(f'Epoch {epoch}/2 | loss={avg_loss:.4f} | val_auc={auc_val:.4f}')

print(f'\nLoss: {losses[0]:.4f} -> {losses[1]:.4f}',
      '✓' if losses[1] < losses[0] else '(may need more steps)')

/home/gokhuu/miniconda3/envs/birdclef/lib/python3.11/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


Epoch 1/2 | loss=0.5972 | val_auc=0.5386
Epoch 2/2 | loss=0.3058 | val_auc=0.5327

Loss: 0.5972 -> 0.3058 ✓


## Verdict

- [ ] Config loads correctly
- [ ] Model instantiates with expected param count
- [ ] Forward pass produces `(B, 234)` output
- [ ] BCE loss + backward pass work
- [ ] Mini-training runs, loss decreases, AUC computes

**Ready for full training:** `python src/training/train.py configs/baseline.yaml`